# LLM Serving Infrastructure

Serving infrastructure schedules model requests across accelerators while balancing latency, throughput, memory, reliability, and cost.


## What to learn

- Batching combines work so hardware is used efficiently.
- The KV cache stores attention state for previously processed tokens.
- Time to first token and time per output token describe different latency phases.
- Quantization reduces memory and compute at a possible quality cost.

## Decision rule

Choose an engine only after measuring the real workload. Benchmark prompt and output lengths, concurrency, latency percentiles, quality, failure recovery, and cost per successful request—not only peak tokens per second.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# Import the dependencies used by this example.
from time import perf_counter
from langsmith import traceable

@traceable(name="model-serving-request", run_type="llm")
# Define the data shape and small operations before running them.
def measured_inference(prompt: str, model: str = "local:vllm") -> dict:
    started = perf_counter()
    # Replace this block with the OpenAI-compatible vLLM/SGLang client call.
    first_token_at = perf_counter()
    text = "simulated completion"
    finished = perf_counter()
    return {
        "text": text,
        "model": model,
        "ttft_ms": (first_token_at - started) * 1000,
        "latency_ms": (finished - started) * 1000,
        "prompt_tokens": len(prompt.split()),
        "completion_tokens": len(text.split()),
        "prefix_cache_hit": False,
    }

measured_inference("Explain continuous batching in one sentence.")


In [ ]:
"""Static vs continuous (iteration-level) batching simulator + prefix-cache economics.
No API needed — the 'GPU' is a discrete-time model: each decode step takes STEP_MS
and emits one token for every active sequence (up to MAX_BATCH KV-cache slots).
Simplifications: prefill folded into the first step; step time constant in batch size.
"""
# Import the dependencies used by this example.
import random

random.seed(7)
STEP_MS   = 30      # ms per decode iteration (one token per active sequence)
MAX_BATCH = 16      # concurrent sequence slots (KV-cache bound in real engines)
N_REQ     = 400

# ---- workload: Poisson arrivals, lognormal output lengths (chat-like traffic)
reqs, t = [], 0.0
for _ in range(N_REQ):
    t += random.expovariate(1 / 300)                                # mean 300 ms gap
    out = min(512, max(4, int(random.lognormvariate(4.2, 0.9))))    # median ~66 tok
    reqs.append({"arrival": t, "out": out})
TOTAL_TOKENS = sum(r["out"] for r in reqs)

# Define the data shape and small operations before running them.
def static_batching():
    """Fill a batch of MAX_BATCH, run until the LONGEST member finishes, repeat."""
    finish, clock, used, padded, i = [0.0] * N_REQ, 0.0, 0, 0, 0
    while i < N_REQ:
        batch = range(i, min(i + MAX_BATCH, N_REQ)); i += len(batch)
        clock = max(clock, max(reqs[j]["arrival"] for j in batch))  # wait to fill batch
        steps = max(reqs[j]["out"] for j in batch)                  # head-of-line block
        for j in batch:
            finish[j] = clock + reqs[j]["out"] * STEP_MS
        clock  += steps * STEP_MS
        used   += sum(reqs[j]["out"] for j in batch)                # useful slot-steps
        padded += len(batch) * steps                                # incl. idle padding
    return finish, used / padded, clock

def continuous_batching():
    """Iteration-level: evict finished sequences and admit waiting ones every step."""
    finish, active, clock, nxt = [0.0] * N_REQ, {}, 0.0, 0
    busy, capacity = 0, 0
    while nxt < N_REQ or active:
        while nxt < N_REQ and reqs[nxt]["arrival"] <= clock and len(active) < MAX_BATCH:
            active[nxt] = reqs[nxt]["out"]; nxt += 1
        if not active:
            clock = reqs[nxt]["arrival"]; continue
        clock += STEP_MS
        busy += len(active); capacity += MAX_BATCH
        for j in list(active):
            active[j] -= 1
            if active[j] == 0:
                finish[j] = clock; del active[j]
    return finish, busy / capacity, clock

def report(name, finish, util, makespan_ms):
    lat = sorted(f - r["arrival"] for r, f in zip(reqs, finish))
    pct = lambda q: lat[int(q * (len(lat) - 1))] / 1000
    tput = TOTAL_TOKENS / (makespan_ms / 1000)
    print(f"{name:<22} util={util:6.1%}  tput={tput:7.1f} tok/s"
          f"  p50={pct(0.50):7.2f}s  p95={pct(0.95):7.2f}s")
    return tput

print(f"Workload: {N_REQ} requests, {TOTAL_TOKENS:,} output tokens, "
      f"{MAX_BATCH} slots, {STEP_MS} ms/step\n")
print("== Scheduler comparison " + "=" * 47)
s_tput = report("static batching", *static_batching())
c_tput = report("continuous batching", *continuous_batching())
print(f"\ncontinuous vs static: {c_tput / s_tput:.2f}x throughput at identical load\n")

# ---- prefix-cache economics: hosted-API pricing model (per 1K requests) ----
SYS, USER, OUT = 6000, 300, 400      # tokens: shared system+tools, unique input, output
IN_P, OUT_P = 3.00, 15.00            # $/M tokens, Sonnet-class list pricing
WRITE_M, READ_M = 1.25, 0.10         # Anthropic-style: 1.25x cache write, 0.1x read

def cost_per_request(hit_rate):
    hit  = (SYS * READ_M  + USER) * IN_P / 1e6 + OUT * OUT_P / 1e6
    miss = (SYS * WRITE_M + USER) * IN_P / 1e6 + OUT * OUT_P / 1e6
    return hit_rate * hit + (1 - hit_rate) * miss

base = ((SYS + USER) * IN_P + OUT * OUT_P) / 1e6      # caching disabled entirely
print("== Prefix-cache economics " + "=" * 45)
print(f"{'hit rate':>8} | {'$ / 1K req':>10} | {'eff $/M input':>13} | {'saving':>7}")
for h in (0.0, 0.5, 0.9, 0.99):
    c = cost_per_request(h)
    eff_in = (c - OUT * OUT_P / 1e6) / (SYS + USER) * 1e6
    print(f"{h:>8.0%} | {1000 * c:>10.2f} | {eff_in:>13.2f} | {1 - c / base:>7.1%}")
print(f"\nno-cache baseline: ${1000 * base:.2f} per 1K requests "
      f"(input list price ${IN_P:.2f}/M)")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
